# COVID — limpieza, preprocesamiento y comparación de modelos
**Notebook independiente de vinos: `02_covid.ipynb`.**

Objetivo de esta primera versión: al terminar el día t en Bolivia, predecir los **casos nuevos reportados del día t+1**, usando únicamente información conocida hasta t. Es un ejercicio histórico de regresión, no diagnóstico de pacientes ni predicción actual de la pandemia.

Elegimos Bolivia como alcance inicial por cercanía al contexto del equipo y para mantener una unidad geográfica coherente, no por sus resultados. No hay un modelo universalmente mejor para COVID: dependerá del objetivo, las variables y las fechas.

Se comparan dos regresiones, dos modelos de árboles y dos SVR. La elección se hace por MAE de validación temporal, antes de consultar el último 20%.

**Actualización: siete modelos candidatos, incluida RNA_MLP; curvas de pérdida y gráficas individuales para todos los modelos.**


## 1. ¿Qué archivo elegir?
Se revisaron los seis archivos suministrados. No se deben concatenar sin más: contienen medidas y niveles geográficos diferentes o solapados.

| Archivo | Filas × columnas | Contenido / uso |
|---|---:|---|
| `covid_19_data.csv` | 306429 × 8 | Reportes por fecha y región con confirmados, muertes y recuperados. Flexible, pero exige revisar nombres y niveles territoriales. |
| `time_series_covid_19_confirmed.csv` | 276 × 498 | Acumulados de confirmados; 494 columnas de fechas. **Elegido para pronóstico de casos.** |
| `time_series_covid_19_deaths.csv` | 276 × 498 | Acumulados de muertes; adecuado para otro objetivo. |
| `time_series_covid_19_recovered.csv` | 261 × 498 | Acumulados de recuperados; diferente cobertura de regiones. |
| `time_series_covid_19_confirmed_US.csv` | 3342 × 505 | Confirmados con detalle territorial de Estados Unidos. |
| `time_series_covid_19_deaths_US.csv` | 3342 × 506 | Muertes en Estados Unidos, incluyendo población. |

Los archivos de series abarcan 22/01/2020–29/05/2021. El nombre Province/State vacío no significa cero casos: puede representar un país reportado sin subdivisión. En el archivo elegido hay 190 provincias vacías y dos coordenadas Lat/Long vacías; no hay nulos en los conteos diarios. Para Bolivia hay una sola fila y 494 días.

El archivo largo no tiene duplicados exactos considerando SNo; al excluir ese identificador aparece una repetición exacta. Esto muestra por qué no basta buscar duplicados en todas las columnas.

El formato es compatible con series de Johns Hopkins, pero la procedencia exacta de este comprimido debe documentarse con el enlace original que usó el equipo.

## 2. Sobre el requisito de 75%
**No existe un 75% universal para todos los modelos.** Accuracy es proporción de clasificaciones correctas; precision es otra métrica de clasificación. En regresión calculamos MAE, MSE, RMSE y R². R²=0.75 no significa acertar el 75% de los días.

Antes de afirmar que se cumple el requisito, preguntar a la docente: «¿El 75% se refiere a accuracy de clasificación, precision, R² ≥ 0.75 o predicciones dentro de una tolerancia definida?»

No convertimos artificialmente este objetivo en categorías ni reportamos 1−MAPE como accuracy para alcanzar una nota. Si se exige clasificación habrá que definir otro objetivo justificado. La revisión de vinos sigue pendiente; este notebook no modifica sus resultados.

## 3. Preparar el proyecto
Primero subir este notebook y el CSV elegido a GitHub. No se necesita Drive. Si la instalación cambia librerías, reiniciar el kernel cuando se indique y ejecutar de nuevo.

In [ ]:
# Path permite trabajar con rutas sin escribir la ruta personal de cada compañero.
from pathlib import Path
import os, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

REPO_URL = "https://github.com/madahi-is/Machine-learning.git"
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# Elegimos la ubicación según dónde se está ejecutando el notebook.
if EN_COLAB:
    destino = Path('/content/Machine-learning-covid')
    if not destino.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(destino)], check=True)
    # Cambiar a la raíz permite usar rutas relativas como data/raw/...
    os.chdir(destino)
else:
    inicio = Path.cwd().resolve()
    raiz = next((p for p in [inicio, *inicio.parents]
                 if (p / 'data/raw/time_series_covid_19_confirmed.csv').exists()), None)
    if raiz is None:
        raise FileNotFoundError('Abre el notebook dentro de tu proyecto Machine-learning.')
    os.chdir(raiz)

# Versiones utilizadas para comprobar esta entrega.
requeridos = {'numpy': '2.3.5', 'pandas': '2.2.3',
              'scikit-learn': '1.8.0', 'matplotlib': '3.10.8', 'seaborn': '0.13.2'}
pendientes = []
for paquete, esperado in requeridos.items():
    try:
        instalado = version(paquete)
    except PackageNotFoundError:
        instalado = None
    if instalado != esperado:
        pendientes.append(f'{paquete}=={esperado}')
# Solo instalamos cuando la versión requerida no está disponible.
if pendientes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pendientes], check=True)
    raise SystemExit('Paquetes instalados. Reinicia la sesión y vuelve a ejecutar esta celda antes de continuar.')
print('Proyecto:', Path.cwd())
print('Versiones:', {p: version(p) for p in requeridos})

## 4. Importar librerías y leer el original

In [ ]:
# pandas maneja tablas; numpy, operaciones numéricas; matplotlib y seaborn, gráficos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from time import perf_counter

# Esta ruta es igual para todos los compañeros y no depende de Drive.
ruta = Path('data/raw/time_series_covid_19_confirmed.csv')
data = pd.read_csv(ruta)
data.columns = data.columns.str.strip()
print('Dimensiones:', data.shape)
# info permite comprobar tipos y nulos; las fechas son columnas en el original.
data.info()
display(data.iloc[:5, :10])
display(data.describe().T.head(10))

## 5. Revisar duplicados, nulos y claves
No eliminamos filas por tener coordenadas o Province/State vacíos: no los necesitamos para este objetivo. Antes de agregar países se debe comprobar que no se mezclan totales nacionales con subtotales.

In [ ]:
# Conservar data intacto. Solo se eliminan filas completamente idénticas.
duplicados = int(data.duplicated().sum())
data_limpia = data.drop_duplicates().copy()
print('Duplicados exactos:', duplicados)
print('Filas después:', len(data_limpia))
print('Nulos en metadatos:')
print(data_limpia[['Province/State', 'Country/Region', 'Lat', 'Long']].isna().sum())

# Las columnas con fechas se identifican por su formato, no por posición fija.
import re
columnas_fechas = [c for c in data_limpia.columns if re.fullmatch(r'\d{1,2}/\d{1,2}/\d{2}', c)]
fechas = pd.to_datetime(columnas_fechas, format='%m/%d/%y', errors='raise')
assert len(fechas) > 30 and not fechas.duplicated().any()
print('Fechas:', len(fechas), fechas.min().date(), fechas.max().date())
print('Nulos en conteos:', int(data_limpia[columnas_fechas].isna().sum().sum()))

# Comprobar la clave territorial; no sumamos filas conflictivas silenciosamente.
clave = data_limpia[['Country/Region', 'Province/State']].fillna('SIN_SUBDIVISION')
if clave.duplicated().any():
    raise ValueError('Claves territoriales repetidas: revisar antes de agregar.')

## 6. Seleccionar Bolivia y transformar fechas de columnas a filas
Una fila de origen no significa una sola observación para el modelo: contiene 494 valores diarios. Convertimos esa fila en una serie con una fecha por fila. Esta versión requiere un registro nacional único; no se generaliza sumando regiones a ciegas.

In [ ]:
PAIS = 'Bolivia'  # Alcance elegido antes de mirar métricas.
seleccion = data_limpia.loc[data_limpia['Country/Region'].eq(PAIS)].copy()
if len(seleccion) != 1:
    raise ValueError('Se esperaba una fila nacional. Revisar agregación geográfica.')

# melt convierte las columnas de fechas en dos columnas: fecha y acumulado.
serie = seleccion.melt(value_vars=columnas_fechas,
                       var_name='fecha', value_name='confirmados_acumulados')
serie['fecha'] = pd.to_datetime(serie['fecha'], format='%m/%d/%y', errors='raise')
serie['confirmados_acumulados'] = pd.to_numeric(serie['confirmados_acumulados'], errors='raise')
serie = serie.sort_values('fecha').set_index('fecha')

# Comprobar continuidad diaria, valores finitos, enteros y no negativos.
assert serie.index.is_unique
assert serie.index.equals(pd.date_range(serie.index.min(), serie.index.max(), freq='D'))
valores = serie['confirmados_acumulados'].to_numpy()
if not np.isfinite(valores).all() or (valores < 0).any() or (valores % 1 != 0).any():
    raise ValueError('Acumulados inválidos: investigar, no rellenar a ciegas.')
print('Serie nacional:', serie.shape)
display(serie.head())
display(serie.tail())

## 7. Obtener casos nuevos reportados
Diferencia entre el acumulado de hoy y ayer. No son necesariamente infecciones ocurridas ese día: pueden reflejar retrasos de reporte. Los acumulados pueden dar R² muy alto por tendencia; predecimos cambios diarios para una tarea más informativa.

In [ ]:
# diff calcula acumulado(t) - acumulado(t-1). La primera diferencia es desconocida.
serie['nuevos'] = serie['confirmados_acumulados'].diff()
print('Diferencias negativas:', int(serie['nuevos'].lt(0).sum()))

# Un descenso puede ser una revisión administrativa; no convertirlo en cero sin evidencia.
if serie['nuevos'].lt(0).any():
    raise ValueError('Se encontraron revisiones negativas: requieren tratamiento documentado.')

# Iniciamos en el primer reporte positivo para no incluir el periodo previo sin casos.
# Esta regla se fija por el significado de la serie, no para mejorar una métrica.
inicio = serie.index[serie['confirmados_acumulados'].gt(0)][0]
serie_modelado = serie.loc[inicio:].copy()
if serie_modelado['nuevos'].isna().any():
    raise ValueError('Falta un valor diario: no interpolar utilizando el futuro.')
print('Inicio con casos:', inicio.date(), 'Días:', len(serie_modelado))
print('Días con reporte cero:', int(serie_modelado['nuevos'].eq(0).sum()))
# Un cero observado se conserva, no se sustituye por la media.
display(serie_modelado.describe().T)

## 8. Construir las entradas y la respuesta
Al final del día t conocemos los casos de hoy y de días anteriores. Buscamos los casos de mañana. Las medias son retrospectivas: no se usa rolling centrado. La fecha objetivo solo se guarda para identificar resultados, no entra como predictor numérico.

In [ ]:
tabla = pd.DataFrame(index=serie_modelado.index)
# lag_0 representa lo observado hoy; lag_6 será la referencia de igual día de semana para mañana.
for retraso in [0, 1, 2, 6, 7, 13]:
    tabla[f'lag_{retraso}'] = serie_modelado['nuevos'].shift(retraso)
# Ventanas que terminan en hoy, sin usar observaciones futuras.
tabla['media_7'] = serie_modelado['nuevos'].rolling(7).mean()
tabla['media_14'] = serie_modelado['nuevos'].rolling(14).mean()
# El calendario de mañana es conocido hoy y puede captar ciclos de reporte.
fecha_objetivo = tabla.index + pd.Timedelta(days=1)
tabla['dia_sin'] = np.sin(2*np.pi*fecha_objetivo.dayofweek/7)
tabla['dia_cos'] = np.cos(2*np.pi*fecha_objetivo.dayofweek/7)
# shift(-1) SOLO se utiliza para construir y, nunca como característica.
tabla['objetivo_manana'] = serie_modelado['nuevos'].shift(-1)
tabla['fecha_objetivo'] = fecha_objetivo

# Los primeros días no tienen todo el historial y el último no tiene respuesta conocida.
filas_antes = len(tabla)
tabla = tabla.dropna().copy()
print('Filas excluidas por historial/objetivo no disponible:', filas_antes-len(tabla))
print('Observaciones supervisadas:', len(tabla))
display(tabla.head())

## 9. Separación cronológica
No usamos train_test_split aleatorio. Reservamos el 20% final y dejamos una fila de separación para que el último objetivo de entrenamiento sea anterior al primer origen de prueba. La validación usará TimeSeriesSplit con gap=1.

In [ ]:
X = tabla.drop(columns=['objetivo_manana', 'fecha_objetivo'])
y = tabla['objetivo_manana']
corte = int(len(tabla)*0.8)
# Se excluye del entrenamiento una fila justo antes del inicio de prueba (gap).
X_train, X_test = X.iloc[:corte-1].copy(), X.iloc[corte:].copy()
y_train, y_test = y.iloc[:corte-1].copy(), y.iloc[corte:].copy()
assert tabla.loc[X_train.index[-1], 'fecha_objetivo'] < X_test.index[0]
print('Train:', X_train.shape, X_train.index.min().date(), X_train.index.max().date())
print('Test:', X_test.shape, X_test.index.min().date(), X_test.index.max().date())
print('Fila de separación:', tabla.index[corte-1].date())

from sklearn.model_selection import TimeSeriesSplit
cv = TimeSeriesSplit(n_splits=5, gap=1)
for numero, (tr, va) in enumerate(cv.split(X_train), start=1):
    assert X_train.index[tr[-1]] + pd.Timedelta(days=1) < X_train.index[va[0]]
    print(f'Fold {numero}: entrenar {len(tr)} días; validar {len(va)} días; '
          f'inicio validación {X_train.index[va[0]].date()}')

## 10. Atípicos, correlación y normalización
Detectamos extremos en entrenamiento con IQR, pero **no recortamos** automáticamente picos epidémicos. A diferencia de vinos, podrían ser justamente los valores que interesa anticipar. No se eliminan días repetidos en sus valores: dos días con igual número de casos son observaciones diferentes.

In [ ]:
# Diagnóstico de la variable de casos de hoy, solo sobre entrenamiento.
q1, q3 = X_train['lag_0'].quantile([0.25, 0.75])
iqr = q3-q1
lim_inf, lim_sup = q1-1.5*iqr, q3+1.5*iqr
extremos = (X_train['lag_0'] < lim_inf) | (X_train['lag_0'] > lim_sup)
print('Límites IQR:', lim_inf, lim_sup, 'Valores fuera:', int(extremos.sum()))
print('Valores modificados por capping: 0')

# Correlaciones en entrenamiento. No justifican relaciones causales.
fig_corr, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(X_train.join(y_train).corr(), annot=True, fmt='.2f',
            cmap='RdBu_r', vmin=-1, vmax=1, annot_kws={'fontsize':8}, ax=ax)
ax.set_title('COVID Bolivia: correlación en entrenamiento')
fig_corr.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Vista didáctica: aprender mínimos/máximos con train, no con test.
escalador_demo = MinMaxScaler()
X_train_normalizado = pd.DataFrame(escalador_demo.fit_transform(X_train),
                                  index=X_train.index, columns=X_train.columns)
display(X_train_normalizado.head())
# No pasar esta matriz a CV: cada pipeline ajustará un escalador nuevo por fold.
# Un futuro valor puede quedar fuera de [0,1]; no se reajusta con test.

# Estandarizamos y dentro de cada entrenamiento para que C y epsilon de SVR
# operen en una escala razonable. predict invierte la transformación a casos.
# Aplicamos el mismo esquema a todos los modelos; métricas siempre en casos.
scoring = {'MAE':'neg_mean_absolute_error', 'MSE':'neg_mean_squared_error',
           'RMSE':'neg_root_mean_squared_error', 'R2':'r2'}
modelos, resultados_cv, resultados_test = {}, {}, {}
predicciones = pd.DataFrame({'fecha_objetivo':tabla.loc[X_test.index,'fecha_objetivo'],
                            'real':y_test}, index=X_test.index)

## 11. Referencias sencillas
Comparamos con «mañana igual a hoy» y «mañana igual al mismo día de la semana pasada». Un modelo complejo puede ser peor que estas reglas. También son candidatos al elegir la mejor solución por validación.

In [ ]:
# Las referencias no se entrenan: seleccionan información ya conocida.
for nombre, columna in [('Persistencia', 'lag_0'), ('Semanal', 'lag_6')]:
    errores = []
    for tr, va in cv.split(X_train):
        errores.append(mean_absolute_error(y_train.iloc[va], X_train.iloc[va][columna]))
    resultados_cv[nombre] = {'modelo':nombre, 'area':'Referencia',
                            'MAE_val_CV':np.mean(errores), 'MAE_val_std':np.std(errores)}
display(pd.DataFrame(resultados_cv.values()))

# **Regresiones**

## Lineal
Regresión lineal múltiple: combina historial reciente y calendario. Puede extrapolar, pero no representa cualquier relación no lineal.
Primero validamos dentro de entrenamiento; luego ajustamos con todo train. Los tiempos son orientativos y dependen del equipo.

In [ ]:
from sklearn.linear_model import LinearRegression
# Cada modelo tiene su propio escalador y ajuste de la escala del objetivo.
modelo_lineal = TransformedTargetRegressor(
    regressor=Pipeline([('normalizar', MinMaxScaler()), ('modelo', LinearRegression())]),
    transformer=StandardScaler()
)
# cross_validate clona todo el proceso: no comparte parámetros entre folds.
scores_lineal = cross_validate(modelo_lineal, X_train, y_train, cv=cv,
                              scoring=scoring, return_train_score=True, error_score='raise')
# Entrenar el modelo definitivo solo con el periodo de entrenamiento.
inicio_reloj = perf_counter()
modelo_lineal.fit(X_train, y_train)
segundos_lineal = perf_counter() - inicio_reloj
modelos['Lineal'] = modelo_lineal

### Métricas de entrenamiento y validación: Lineal
Una brecha grande puede indicar sobreajuste, pero también cambios temporales en el proceso. No se declara con un umbral arbitrario. El R² medio por folds no equivale al R² de todas las predicciones concatenadas.

In [ ]:
# test_* significa validación de cada fold; todavía no se ha evaluado X_test.
resultados_cv['Lineal'] = {'modelo':'Lineal', 'area':'Regresiones',
    'MAE_train_CV':-scores_lineal['train_MAE'].mean(),
    'MAE_val_CV':-scores_lineal['test_MAE'].mean(),
    'MAE_val_std':scores_lineal['test_MAE'].std(),
    'MSE_val_CV':-scores_lineal['test_MSE'].mean(),
    'RMSE_val_CV':-scores_lineal['test_RMSE'].mean(),
    'R2_val_CV':scores_lineal['test_R2'].mean(),
    'segundos_fit_final':segundos_lineal,
    'segundos_fit_fold':scores_lineal['fit_time'].mean()}
display(pd.DataFrame([resultados_cv['Lineal']]).round(4))

## Ridge
Ridge: penaliza coeficientes grandes. alpha=1.0 fija la regularización. Es útil como candidato con rezagos correlacionados; no garantiza ganar.
Primero validamos dentro de entrenamiento; luego ajustamos con todo train. Los tiempos son orientativos y dependen del equipo.

In [ ]:
from sklearn.linear_model import Ridge
# Cada modelo tiene su propio escalador y ajuste de la escala del objetivo.
modelo_ridge = TransformedTargetRegressor(
    regressor=Pipeline([('normalizar', MinMaxScaler()), ('modelo', Ridge(alpha=1.0))]),
    transformer=StandardScaler()
)
# cross_validate clona todo el proceso: no comparte parámetros entre folds.
scores_ridge = cross_validate(modelo_ridge, X_train, y_train, cv=cv,
                              scoring=scoring, return_train_score=True, error_score='raise')
# Entrenar el modelo definitivo solo con el periodo de entrenamiento.
inicio_reloj = perf_counter()
modelo_ridge.fit(X_train, y_train)
segundos_ridge = perf_counter() - inicio_reloj
modelos['Ridge'] = modelo_ridge

### Métricas de entrenamiento y validación: Ridge
Una brecha grande puede indicar sobreajuste, pero también cambios temporales en el proceso. No se declara con un umbral arbitrario. El R² medio por folds no equivale al R² de todas las predicciones concatenadas.

In [ ]:
# test_* significa validación de cada fold; todavía no se ha evaluado X_test.
resultados_cv['Ridge'] = {'modelo':'Ridge', 'area':'Regresiones',
    'MAE_train_CV':-scores_ridge['train_MAE'].mean(),
    'MAE_val_CV':-scores_ridge['test_MAE'].mean(),
    'MAE_val_std':scores_ridge['test_MAE'].std(),
    'MSE_val_CV':-scores_ridge['test_MSE'].mean(),
    'RMSE_val_CV':-scores_ridge['test_RMSE'].mean(),
    'R2_val_CV':scores_ridge['test_R2'].mean(),
    'segundos_fit_final':segundos_ridge,
    'segundos_fit_fold':scores_ridge['fit_time'].mean()}
display(pd.DataFrame([resultados_cv['Ridge']]).round(4))

# **Árboles**

## Arbol
Árbol: max_depth=5 limita profundidad y min_samples_leaf=5 evita hojas diminutas. No extrapola bien fuera de valores objetivo observados.
Primero validamos dentro de entrenamiento; luego ajustamos con todo train. Los tiempos son orientativos y dependen del equipo.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
# Cada modelo tiene su propio escalador y ajuste de la escala del objetivo.
modelo_arbol = TransformedTargetRegressor(
    regressor=Pipeline([('normalizar', MinMaxScaler()), ('modelo', DecisionTreeRegressor(max_depth=5, min_samples_leaf=5, random_state=42))]),
    transformer=StandardScaler()
)
# cross_validate clona todo el proceso: no comparte parámetros entre folds.
scores_arbol = cross_validate(modelo_arbol, X_train, y_train, cv=cv,
                              scoring=scoring, return_train_score=True, error_score='raise')
# Entrenar el modelo definitivo solo con el periodo de entrenamiento.
inicio_reloj = perf_counter()
modelo_arbol.fit(X_train, y_train)
segundos_arbol = perf_counter() - inicio_reloj
modelos['Arbol'] = modelo_arbol

### Métricas de entrenamiento y validación: Arbol
Una brecha grande puede indicar sobreajuste, pero también cambios temporales en el proceso. No se declara con un umbral arbitrario. El R² medio por folds no equivale al R² de todas las predicciones concatenadas.

In [ ]:
# test_* significa validación de cada fold; todavía no se ha evaluado X_test.
resultados_cv['Arbol'] = {'modelo':'Arbol', 'area':'Árboles',
    'MAE_train_CV':-scores_arbol['train_MAE'].mean(),
    'MAE_val_CV':-scores_arbol['test_MAE'].mean(),
    'MAE_val_std':scores_arbol['test_MAE'].std(),
    'MSE_val_CV':-scores_arbol['test_MSE'].mean(),
    'RMSE_val_CV':-scores_arbol['test_RMSE'].mean(),
    'R2_val_CV':scores_arbol['test_R2'].mean(),
    'segundos_fit_final':segundos_arbol,
    'segundos_fit_fold':scores_arbol['fit_time'].mean()}
display(pd.DataFrame([resultados_cv['Arbol']]).round(4))

## RandomForest
Random Forest: promedio de 200 árboles; hojas de al menos tres muestras. n_jobs=1 limita recursos. También tiene límites para extrapolar.
Primero validamos dentro de entrenamiento; luego ajustamos con todo train. Los tiempos son orientativos y dependen del equipo.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
# Cada modelo tiene su propio escalador y ajuste de la escala del objetivo.
modelo_bosque = TransformedTargetRegressor(
    regressor=Pipeline([('normalizar', MinMaxScaler()), ('modelo', RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=1))]),
    transformer=StandardScaler()
)
# cross_validate clona todo el proceso: no comparte parámetros entre folds.
scores_bosque = cross_validate(modelo_bosque, X_train, y_train, cv=cv,
                              scoring=scoring, return_train_score=True, error_score='raise')
# Entrenar el modelo definitivo solo con el periodo de entrenamiento.
inicio_reloj = perf_counter()
modelo_bosque.fit(X_train, y_train)
segundos_bosque = perf_counter() - inicio_reloj
modelos['RandomForest'] = modelo_bosque

### Métricas de entrenamiento y validación: RandomForest
Una brecha grande puede indicar sobreajuste, pero también cambios temporales en el proceso. No se declara con un umbral arbitrario. El R² medio por folds no equivale al R² de todas las predicciones concatenadas.

In [ ]:
# test_* significa validación de cada fold; todavía no se ha evaluado X_test.
resultados_cv['RandomForest'] = {'modelo':'RandomForest', 'area':'Árboles',
    'MAE_train_CV':-scores_bosque['train_MAE'].mean(),
    'MAE_val_CV':-scores_bosque['test_MAE'].mean(),
    'MAE_val_std':scores_bosque['test_MAE'].std(),
    'MSE_val_CV':-scores_bosque['test_MSE'].mean(),
    'RMSE_val_CV':-scores_bosque['test_RMSE'].mean(),
    'R2_val_CV':scores_bosque['test_R2'].mean(),
    'segundos_fit_final':segundos_bosque,
    'segundos_fit_fold':scores_bosque['fit_time'].mean()}
display(pd.DataFrame([resultados_cv['RandomForest']]).round(4))

# **SVM**

## SVR_lineal
SVR lineal: C=1 regula la penalización y epsilon=0.1 la tolerancia en unidades estandarizadas del objetivo.
Primero validamos dentro de entrenamiento; luego ajustamos con todo train. Los tiempos son orientativos y dependen del equipo.

In [ ]:
from sklearn.svm import SVR
# Cada modelo tiene su propio escalador y ajuste de la escala del objetivo.
modelo_svr_lineal = TransformedTargetRegressor(
    regressor=Pipeline([('normalizar', MinMaxScaler()), ('modelo', SVR(kernel='linear', C=1.0, epsilon=0.1))]),
    transformer=StandardScaler()
)
# cross_validate clona todo el proceso: no comparte parámetros entre folds.
scores_svr_lineal = cross_validate(modelo_svr_lineal, X_train, y_train, cv=cv,
                              scoring=scoring, return_train_score=True, error_score='raise')
# Entrenar el modelo definitivo solo con el periodo de entrenamiento.
inicio_reloj = perf_counter()
modelo_svr_lineal.fit(X_train, y_train)
segundos_svr_lineal = perf_counter() - inicio_reloj
modelos['SVR_lineal'] = modelo_svr_lineal

### Métricas de entrenamiento y validación: SVR_lineal
Una brecha grande puede indicar sobreajuste, pero también cambios temporales en el proceso. No se declara con un umbral arbitrario. El R² medio por folds no equivale al R² de todas las predicciones concatenadas.

In [ ]:
# test_* significa validación de cada fold; todavía no se ha evaluado X_test.
resultados_cv['SVR_lineal'] = {'modelo':'SVR_lineal', 'area':'SVM',
    'MAE_train_CV':-scores_svr_lineal['train_MAE'].mean(),
    'MAE_val_CV':-scores_svr_lineal['test_MAE'].mean(),
    'MAE_val_std':scores_svr_lineal['test_MAE'].std(),
    'MSE_val_CV':-scores_svr_lineal['test_MSE'].mean(),
    'RMSE_val_CV':-scores_svr_lineal['test_RMSE'].mean(),
    'R2_val_CV':scores_svr_lineal['test_R2'].mean(),
    'segundos_fit_final':segundos_svr_lineal,
    'segundos_fit_fold':scores_svr_lineal['fit_time'].mean()}
display(pd.DataFrame([resultados_cv['SVR_lineal']]).round(4))

## SVR_RBF
SVR RBF: relación no lineal; C=10 y gamma=scale son una configuración inicial, no seleccionada usando test.
Primero validamos dentro de entrenamiento; luego ajustamos con todo train. Los tiempos son orientativos y dependen del equipo.

In [ ]:
from sklearn.svm import SVR
# Cada modelo tiene su propio escalador y ajuste de la escala del objetivo.
modelo_svr_rbf = TransformedTargetRegressor(
    regressor=Pipeline([('normalizar', MinMaxScaler()), ('modelo', SVR(kernel='rbf', C=10.0, epsilon=0.1, gamma='scale'))]),
    transformer=StandardScaler()
)
# cross_validate clona todo el proceso: no comparte parámetros entre folds.
scores_svr_rbf = cross_validate(modelo_svr_rbf, X_train, y_train, cv=cv,
                              scoring=scoring, return_train_score=True, error_score='raise')
# Entrenar el modelo definitivo solo con el periodo de entrenamiento.
inicio_reloj = perf_counter()
modelo_svr_rbf.fit(X_train, y_train)
segundos_svr_rbf = perf_counter() - inicio_reloj
modelos['SVR_RBF'] = modelo_svr_rbf

### Métricas de entrenamiento y validación: SVR_RBF
Una brecha grande puede indicar sobreajuste, pero también cambios temporales en el proceso. No se declara con un umbral arbitrario. El R² medio por folds no equivale al R² de todas las predicciones concatenadas.

In [ ]:
# test_* significa validación de cada fold; todavía no se ha evaluado X_test.
resultados_cv['SVR_RBF'] = {'modelo':'SVR_RBF', 'area':'SVM',
    'MAE_train_CV':-scores_svr_rbf['train_MAE'].mean(),
    'MAE_val_CV':-scores_svr_rbf['test_MAE'].mean(),
    'MAE_val_std':scores_svr_rbf['test_MAE'].std(),
    'MSE_val_CV':-scores_svr_rbf['test_MSE'].mean(),
    'RMSE_val_CV':-scores_svr_rbf['test_RMSE'].mean(),
    'R2_val_CV':scores_svr_rbf['test_R2'].mean(),
    'segundos_fit_final':segundos_svr_rbf,
    'segundos_fit_fold':scores_svr_rbf['fit_time'].mean()}
display(pd.DataFrame([resultados_cv['SVR_RBF']]).round(4))

# Redes neuronales: perceptrón multicapa para regresión
Basado en el material de clase, páginas 13–14: perceptrón multicapa y retropropagación. Usamos `MLPRegressor` de scikit-learn, sin instalar TensorFlow.

**Arquitectura:** 10 → 16 → 8 → 1. Las capas ocultas usan ReLU y la salida es lineal. La red aprende relaciones no lineales; una arquitectura pequeña limita la complejidad con pocos ejemplos. Esta es una configuración inicial fijada antes de esta evaluación, no una arquitectura óptima demostrada.

La red recibe rezagos, medias retrospectivas y calendario; no es recurrente ni una LSTM. Conservamos TimeSeriesSplit y el gap. Desactivamos validación interna aleatoria y mezcla de lotes. Predice un día adelante con historia real actualizada, no todo el periodo desde una sola fecha.

**Cómo aprende:** propagación hacia adelante → comparación con el objetivo → retropropagación de gradientes → actualización Adam. Se repite por épocas. No implementamos manualmente las derivadas: la librería realiza el entrenamiento.

**Evaluación:** misma partición y mismos folds que los demás modelos. Selección por MAE de validación. No prometemos que una red sea mejor. Como este test ya se examinó en versiones anteriores, esta ampliación es exploratoria: para una confirmación independiente se necesitarían datos nuevos no utilizados.


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from time import perf_counter

# Perceptrón multicapa: cada neurona combina entradas, pesos y un sesgo.
# ReLU(x)=max(0,x) permite aprender relaciones no lineales en las capas ocultas.
# La salida usa identidad: devuelve un número continuo, no una clase.
# Adam actualiza pesos usando gradientes calculados por retropropagación.
red = MLPRegressor(
    hidden_layer_sizes=(16, 8),  # Dos capas ocultas; la entrada se infiere de X.
    activation='relu', solver='adam',
    alpha=0.01,                 # Penalización L2: limita pesos excesivos.
    learning_rate_init=0.001,   # Tamaño inicial de las actualizaciones.
    batch_size=32,             # Ejemplos por actualización de pesos.
    max_iter=1500,             # Máximo de épocas; no garantiza convergencia.
    tol=1e-5, n_iter_no_change=30,
    early_stopping=False,      # No crear una validación interna aleatoria.
    shuffle=False, random_state=42
)
# StandardScaler aprende media/desviación de y solo con entrenamiento.
# predict deshace ese escalado: las métricas conservan las unidades originales.
modelo_rna = TransformedTargetRegressor(
    regressor=Pipeline([('normalizar', MinMaxScaler()), ('modelo', red)]),
    transformer=StandardScaler())
# Ajuste final usando únicamente train; test sigue fuera del entrenamiento.
inicio_reloj = perf_counter()
modelo_rna.fit(X_train, y_train)
segundos_rna = perf_counter() - inicio_reloj
modelos['RNA_MLP'] = modelo_rna
pred_train_rna = modelo_rna.predict(X_train)
display(pd.DataFrame({'real': y_train.iloc[:5], 'predicho': pred_train_rna[:5]}))


### Validación de la red
Una pérdida de entrenamiento decreciente no demuestra generalización. Revisar MAE de train frente a validación y referencias.

In [ ]:
# Cada fold clona la red y aprende de nuevo sus escaladores y sus pesos.
# Se reutilizan los mismos folds que en los otros modelos para compararlos.
scores_rna = cross_validate(modelo_rna, X_train, y_train, cv=cv,
    scoring=scoring, return_train_score=True, error_score='raise')
resultados_cv['RNA_MLP'] = {
    'modelo':'RNA_MLP', 'area':'Redes neuronales',
    'MAE_train_CV':-scores_rna['train_MAE'].mean(),
    'MAE_val_CV':-scores_rna['test_MAE'].mean(),
    'MAE_val_std':scores_rna['test_MAE'].std(),
    'MSE_val_CV':-scores_rna['test_MSE'].mean(),
    'RMSE_val_CV':-scores_rna['test_RMSE'].mean(),
    'R2_train_CV':scores_rna['train_R2'].mean(),
    'R2_val_CV':scores_rna['test_R2'].mean(),
    'segundos_fit_final':segundos_rna,
    'segundos_fit_fold':scores_rna['fit_time'].mean()}
display(pd.DataFrame([resultados_cv['RNA_MLP']]).round(4))


### Curva de aprendizaje durante el ajuste
Muestra la pérdida por época del ajuste final sobre train. No usa datos de prueba.

In [ ]:
# Accedemos a la red ajustada, no a su plantilla sin entrenar.
red_ajustada = modelo_rna.regressor_.named_steps['modelo']
carpeta_graficas = Path('results/covid_bolivia/graficas')
carpeta_graficas.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(np.arange(1, len(red_ajustada.loss_curve_)+1), red_ajustada.loss_curve_)
ax.set(xlabel='Época', ylabel='Pérdida de entrenamiento (y estandarizada)',
       title='RNA — evolución del entrenamiento')
ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(carpeta_graficas/'RNA_perdida.png', dpi=150)
plt.show(); plt.close(fig)
# Esta pérdida incluye regularización y NO es MAE, accuracy ni error de validación.
print('Épocas ejecutadas:', red_ajustada.n_iter_)
print('Arquitectura:', [red_ajustada.coefs_[0].shape[0]] +
      [w.shape[1] for w in red_ajustada.coefs_])
print('Parámetros aprendidos:', sum(w.size for w in red_ajustada.coefs_) +
      sum(b.size for b in red_ajustada.intercepts_))
if red_ajustada.n_iter_ >= red_ajustada.max_iter:
    print('Se alcanzó el máximo de épocas: revisar convergencia, sin ajustar con test.')
pd.DataFrame({'epoca':np.arange(1,len(red_ajustada.loss_curve_)+1),
              'perdida_train':red_ajustada.loss_curve_}).to_csv(carpeta_graficas/'RNA_perdida.csv',index=False)


## 12. Elegir por validación antes de evaluar prueba
Se selecciona el mejor algoritmo entre los siete por MAE_CV. Si una referencia tiene menor error, esa regla es la mejor solución en esta comparación. No cambiamos modelos o país para mejorar el test.

In [ ]:
assert len(modelos) == 7 and len(resultados_cv) == 9
comparacion_cv = pd.DataFrame(resultados_cv.values()).sort_values('MAE_val_CV')
mejor_solucion = comparacion_cv.iloc[0]['modelo']
mejor_modelo = comparacion_cv.loc[comparacion_cv.area.ne('Referencia')].iloc[0]['modelo']
display(comparacion_cv.round(4))
print('Mejor algoritmo por CV:', mejor_modelo)
print('Mejor solución, incluidas referencias:', mejor_solucion)

## 13. Evaluación final de un día adelante
Los modelos quedan fijos después de train. En cada origen de test reciben los reportes reales disponibles hasta ese día y predicen el siguiente. Es evaluación retrospectiva de **un paso con actualización de entradas**, no un pronóstico de todo el periodo realizado desde el primer día.

Que las características de un día de test contengan reportes de días anteriores de test es correcto para este escenario: ya se observaron al emitir ese pronóstico. No se usa la respuesta del día que se intenta predecir.

La fuente puede incorporar revisiones históricas hechas después de la fecha de reporte; sin versiones diarias archivadas no se garantiza una simulación exacta en tiempo real. No se redondean ni recortan predicciones para favorecer métricas; se informa cuántas son negativas.

### Lineal: predicciones

In [ ]:
# predict aplica los parámetros ya aprendidos y devuelve resultados en casos.
inicio_reloj = perf_counter()
y_pred_lineal = modelo_lineal.predict(X_test)
tiempo_pred_lineal = perf_counter()-inicio_reloj
predicciones['Lineal'] = y_pred_lineal
print('Primeras cinco predicciones:')
for i in range(5):
    print(f'Fecha {predicciones.fecha_objetivo.iloc[i].date()}: '
          f'real={y_test.iloc[i]:.0f}, predicho={y_pred_lineal[i]:.2f}')

### Lineal: métricas de prueba

In [ ]:
# MAE y RMSE se expresan en casos reportados por día; MSE, en casos al cuadrado.
mse_lineal = mean_squared_error(y_test, y_pred_lineal)
resultados_test['Lineal'] = {'modelo':'Lineal',
    'MAE':mean_absolute_error(y_test,y_pred_lineal), 'MSE':mse_lineal,
    'RMSE':np.sqrt(mse_lineal), 'R2':r2_score(y_test,y_pred_lineal),
    'predicciones_negativas':int((y_pred_lineal<0).sum()),
    'segundos_prediccion':tiempo_pred_lineal}
display(pd.DataFrame([resultados_test['Lineal']]).round(4))
# R2 no se etiqueta como accuracy. Tampoco convertimos el error en "precisión".

### Ridge: predicciones

In [ ]:
# predict aplica los parámetros ya aprendidos y devuelve resultados en casos.
inicio_reloj = perf_counter()
y_pred_ridge = modelo_ridge.predict(X_test)
tiempo_pred_ridge = perf_counter()-inicio_reloj
predicciones['Ridge'] = y_pred_ridge
print('Primeras cinco predicciones:')
for i in range(5):
    print(f'Fecha {predicciones.fecha_objetivo.iloc[i].date()}: '
          f'real={y_test.iloc[i]:.0f}, predicho={y_pred_ridge[i]:.2f}')

### Ridge: métricas de prueba

In [ ]:
# MAE y RMSE se expresan en casos reportados por día; MSE, en casos al cuadrado.
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
resultados_test['Ridge'] = {'modelo':'Ridge',
    'MAE':mean_absolute_error(y_test,y_pred_ridge), 'MSE':mse_ridge,
    'RMSE':np.sqrt(mse_ridge), 'R2':r2_score(y_test,y_pred_ridge),
    'predicciones_negativas':int((y_pred_ridge<0).sum()),
    'segundos_prediccion':tiempo_pred_ridge}
display(pd.DataFrame([resultados_test['Ridge']]).round(4))
# R2 no se etiqueta como accuracy. Tampoco convertimos el error en "precisión".

### Arbol: predicciones

In [ ]:
# predict aplica los parámetros ya aprendidos y devuelve resultados en casos.
inicio_reloj = perf_counter()
y_pred_arbol = modelo_arbol.predict(X_test)
tiempo_pred_arbol = perf_counter()-inicio_reloj
predicciones['Arbol'] = y_pred_arbol
print('Primeras cinco predicciones:')
for i in range(5):
    print(f'Fecha {predicciones.fecha_objetivo.iloc[i].date()}: '
          f'real={y_test.iloc[i]:.0f}, predicho={y_pred_arbol[i]:.2f}')

### Arbol: métricas de prueba

In [ ]:
# MAE y RMSE se expresan en casos reportados por día; MSE, en casos al cuadrado.
mse_arbol = mean_squared_error(y_test, y_pred_arbol)
resultados_test['Arbol'] = {'modelo':'Arbol',
    'MAE':mean_absolute_error(y_test,y_pred_arbol), 'MSE':mse_arbol,
    'RMSE':np.sqrt(mse_arbol), 'R2':r2_score(y_test,y_pred_arbol),
    'predicciones_negativas':int((y_pred_arbol<0).sum()),
    'segundos_prediccion':tiempo_pred_arbol}
display(pd.DataFrame([resultados_test['Arbol']]).round(4))
# R2 no se etiqueta como accuracy. Tampoco convertimos el error en "precisión".

### RandomForest: predicciones

In [ ]:
# predict aplica los parámetros ya aprendidos y devuelve resultados en casos.
inicio_reloj = perf_counter()
y_pred_bosque = modelo_bosque.predict(X_test)
tiempo_pred_bosque = perf_counter()-inicio_reloj
predicciones['RandomForest'] = y_pred_bosque
print('Primeras cinco predicciones:')
for i in range(5):
    print(f'Fecha {predicciones.fecha_objetivo.iloc[i].date()}: '
          f'real={y_test.iloc[i]:.0f}, predicho={y_pred_bosque[i]:.2f}')

### RandomForest: métricas de prueba

In [ ]:
# MAE y RMSE se expresan en casos reportados por día; MSE, en casos al cuadrado.
mse_bosque = mean_squared_error(y_test, y_pred_bosque)
resultados_test['RandomForest'] = {'modelo':'RandomForest',
    'MAE':mean_absolute_error(y_test,y_pred_bosque), 'MSE':mse_bosque,
    'RMSE':np.sqrt(mse_bosque), 'R2':r2_score(y_test,y_pred_bosque),
    'predicciones_negativas':int((y_pred_bosque<0).sum()),
    'segundos_prediccion':tiempo_pred_bosque}
display(pd.DataFrame([resultados_test['RandomForest']]).round(4))
# R2 no se etiqueta como accuracy. Tampoco convertimos el error en "precisión".

### SVR_lineal: predicciones

In [ ]:
# predict aplica los parámetros ya aprendidos y devuelve resultados en casos.
inicio_reloj = perf_counter()
y_pred_svr_lineal = modelo_svr_lineal.predict(X_test)
tiempo_pred_svr_lineal = perf_counter()-inicio_reloj
predicciones['SVR_lineal'] = y_pred_svr_lineal
print('Primeras cinco predicciones:')
for i in range(5):
    print(f'Fecha {predicciones.fecha_objetivo.iloc[i].date()}: '
          f'real={y_test.iloc[i]:.0f}, predicho={y_pred_svr_lineal[i]:.2f}')

### SVR_lineal: métricas de prueba

In [ ]:
# MAE y RMSE se expresan en casos reportados por día; MSE, en casos al cuadrado.
mse_svr_lineal = mean_squared_error(y_test, y_pred_svr_lineal)
resultados_test['SVR_lineal'] = {'modelo':'SVR_lineal',
    'MAE':mean_absolute_error(y_test,y_pred_svr_lineal), 'MSE':mse_svr_lineal,
    'RMSE':np.sqrt(mse_svr_lineal), 'R2':r2_score(y_test,y_pred_svr_lineal),
    'predicciones_negativas':int((y_pred_svr_lineal<0).sum()),
    'segundos_prediccion':tiempo_pred_svr_lineal}
display(pd.DataFrame([resultados_test['SVR_lineal']]).round(4))
# R2 no se etiqueta como accuracy. Tampoco convertimos el error en "precisión".

### SVR_RBF: predicciones

In [ ]:
# predict aplica los parámetros ya aprendidos y devuelve resultados en casos.
inicio_reloj = perf_counter()
y_pred_svr_rbf = modelo_svr_rbf.predict(X_test)
tiempo_pred_svr_rbf = perf_counter()-inicio_reloj
predicciones['SVR_RBF'] = y_pred_svr_rbf
print('Primeras cinco predicciones:')
for i in range(5):
    print(f'Fecha {predicciones.fecha_objetivo.iloc[i].date()}: '
          f'real={y_test.iloc[i]:.0f}, predicho={y_pred_svr_rbf[i]:.2f}')

### SVR_RBF: métricas de prueba

In [ ]:
# MAE y RMSE se expresan en casos reportados por día; MSE, en casos al cuadrado.
mse_svr_rbf = mean_squared_error(y_test, y_pred_svr_rbf)
resultados_test['SVR_RBF'] = {'modelo':'SVR_RBF',
    'MAE':mean_absolute_error(y_test,y_pred_svr_rbf), 'MSE':mse_svr_rbf,
    'RMSE':np.sqrt(mse_svr_rbf), 'R2':r2_score(y_test,y_pred_svr_rbf),
    'predicciones_negativas':int((y_pred_svr_rbf<0).sum()),
    'segundos_prediccion':tiempo_pred_svr_rbf}
display(pd.DataFrame([resultados_test['SVR_RBF']]).round(4))
# R2 no se etiqueta como accuracy. Tampoco convertimos el error en "precisión".

### RNA_MLP: predicciones

In [ ]:
# predict aplica los parámetros ya aprendidos y devuelve resultados en casos.
inicio_reloj = perf_counter()
y_pred_rna = modelo_rna.predict(X_test)
tiempo_pred_rna = perf_counter()-inicio_reloj
predicciones['RNA_MLP'] = y_pred_rna
print('Primeras cinco predicciones:')
for i in range(5):
    print(f'Fecha {predicciones.fecha_objetivo.iloc[i].date()}: '
          f'real={y_test.iloc[i]:.0f}, predicho={y_pred_rna[i]:.2f}')

### RNA_MLP: métricas de prueba

In [ ]:
# MAE y RMSE se expresan en casos reportados por día; MSE, en casos al cuadrado.
mse_rna = mean_squared_error(y_test, y_pred_rna)
resultados_test['RNA_MLP'] = {'modelo':'RNA_MLP',
    'MAE':mean_absolute_error(y_test,y_pred_rna), 'MSE':mse_rna,
    'RMSE':np.sqrt(mse_rna), 'R2':r2_score(y_test,y_pred_rna),
    'predicciones_negativas':int((y_pred_rna<0).sum()),
    'segundos_prediccion':tiempo_pred_rna}
display(pd.DataFrame([resultados_test['RNA_MLP']]).round(4))
# R2 no se etiqueta como accuracy. Tampoco convertimos el error en "precisión".

## 14. Comparación final, eficiencia y límites

In [ ]:
for nombre, columna in [('Persistencia','lag_0'), ('Semanal','lag_6')]:
    pred = X_test[columna].to_numpy()
    predicciones[nombre] = pred
    mse = mean_squared_error(y_test,pred)
    resultados_test[nombre] = {'modelo':nombre, 'MAE':mean_absolute_error(y_test,pred),
        'MSE':mse, 'RMSE':np.sqrt(mse), 'R2':r2_score(y_test,pred),
        'predicciones_negativas':int((pred<0).sum())}
comparacion_test = pd.DataFrame(resultados_test.values())
comparacion_test['elegido_por_cv'] = comparacion_test.modelo.eq(mejor_solucion)
display(comparacion_test.round(4))
print('La selección sigue siendo:', mejor_solucion)
print('El requisito de 75% está pendiente de definición de la métrica por la docente.')

# Eficacia predictiva: errores, R2 y comparación con referencias.
# Eficiencia computacional: segundos de entrenamiento y predicción, medidos una vez.
# Estos tiempos no son un benchmark estable ni comparables entre computadoras.
fig_pred, ax = plt.subplots(figsize=(12,5))
ax.plot(predicciones.fecha_objetivo, y_test.to_numpy(), label='Real', color='#263238')
ax.plot(predicciones.fecha_objetivo, predicciones[mejor_solucion], label=f'Elegido por CV: {mejor_solucion}', alpha=.85)
ax.set(xlabel='Fecha objetivo', ylabel='Casos nuevos reportados', title='Bolivia — evaluación de un día adelante')
ax.legend(); fig_pred.autofmt_xdate(); fig_pred.tight_layout(); plt.show()

## 15. Guardar resultados
La carpeta es independiente de vinos. El archivo normalizado es solo del ajuste final; para nueva CV se parte de X_train sin escalar.

In [ ]:
import json, platform, hashlib
salida = Path('results/covid_bolivia')
salida.mkdir(parents=True, exist_ok=True)
serie.to_csv(salida/'serie_diaria.csv')
tabla.to_csv(salida/'tabla_supervisada.csv')
comparacion_cv.to_csv(salida/'comparacion_cv.csv',index=False)
comparacion_test.to_csv(salida/'comparacion_test.csv',index=False)
predicciones.to_csv(salida/'predicciones_test.csv',index_label='fecha_origen')
fig_corr.savefig(salida/'correlacion_train.png',dpi=150,bbox_inches='tight')
fig_pred.savefig(salida/'predicciones.png',dpi=150,bbox_inches='tight')
# El escalador del algoritmo elegido se ajustó solo sobre train.
escalador_final = modelos[mejor_modelo].regressor_.named_steps['normalizar']
for nombre, datos in [('train',X_train),('test',X_test)]:
    pd.DataFrame(escalador_final.transform(datos),index=datos.index,columns=datos.columns).to_csv(salida/f'X_{nombre}_normalizado.csv')
resumen = {'pais':PAIS,'objetivo':'casos nuevos reportados al dia siguiente',
    'filas_originales':len(data),'duplicados':duplicados,'dias':len(serie),
    'observaciones_supervisadas':len(tabla),'train':len(X_train),'test':len(X_test),'gap':1,
    'mejor_algoritmo_cv':mejor_modelo,'mejor_solucion_cv':mejor_solucion,
    'python':platform.python_version(),'versiones':{p:version(p) for p in requeridos},
    'sha256_csv':hashlib.sha256(ruta.read_bytes()).hexdigest(),
    'criterio_75_por_ciento':'Pendiente de aclarar con la docente'}
(salida/'resumen.json').write_text(json.dumps(resumen,indent=2,ensure_ascii=False),encoding='utf-8')
print('Resultados:',salida.resolve())

## Conclusiones que deben escribir en el informe
- Justificar el archivo y el alcance Bolivia; no generalizar el ganador a otros países, objetivos o periodos.
- Informar que no hubo duplicados exactos, faltantes de conteos ni diferencias negativas en esta serie nacional.
- Explicar que se conservan los ceros observados y los picos, y no se usa información futura para escalar.
- Comparar los seis algoritmos y ambas referencias por MAE_CV, y luego reportar test sin reoptimizarlo.
- Revisar la brecha train/validación y los tiempos; un modelo más complejo no garantiza mayor utilidad.
- No afirmar 75% de accuracy con R² o con 1−MAPE. Aclarar el criterio con la docente.
- Limitaciones: pocos días, reportes irregulares, revisiones históricas, cambios de dinámica y evaluación de un paso.

Referencias: [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html), [R²](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html), [documentación de series JHU](https://github.com/CSSEGISandData/COVID-19/blob/master/csse_covid_19_data/README.md). No se ha verificado el enlace original del comprimido aportado.

## Gráficas de evaluación de TODOS los modelos
Cada punto del primer panel es una observación de prueba. La diagonal representa predicción perfecta (y=x), **no la recta ajustada de regresión**. En el segundo panel, residuo = real − predicho: positivo significa subestimación. Buscar errores próximos a cero sin patrones; estos gráficos no prueban por sí solos ausencia de sobreajuste.

La regresión es múltiple: usa varias entradas y no se puede representar completamente con una sola recta en dos dimensiones. En vinos los puntos se superponen porque las calidades reales son enteras.

In [ ]:
# Reutilizamos solo el dibujo: cada modelo conserva su entrenamiento y métricas.
def graficar_modelo(nombre, pred):
    real = y_test.to_numpy()
    residuo = real - np.asarray(pred)
    fig, axes = plt.subplots(1, 2, figsize=(11,4))
    axes[0].scatter(real, pred, alpha=.4, s=22)
    limites = [min(real.min(), np.min(pred)), max(real.max(), np.max(pred))]
    axes[0].plot(limites, limites, '--', color='black', label='Predicción perfecta: y=x')
    axes[0].set(xlabel='Valor real', ylabel='Predicción', title=nombre+' — reales vs predichos')
    axes[0].legend()
    axes[1].scatter(pred, residuo, alpha=.4, s=22)
    axes[1].axhline(0, color='black', linestyle='--')
    axes[1].set(xlabel='Predicción', ylabel='Residuo: real − predicho', title='Errores de prueba')
    for ax in axes: ax.grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(carpeta_graficas/(nombre+'_diagnostico.png'), dpi=150)
    plt.show(); plt.close(fig)
    # Eje de fechas: permite ver si el modelo sigue los picos y cambios temporales.
    fig, ax = plt.subplots(figsize=(12,4))
    fechas = predicciones['fecha_objetivo']
    ax.plot(fechas, real, label='Real', color='black', linewidth=1.5)
    ax.plot(fechas, pred, label='Predicho', alpha=.85)
    ax.set(title=nombre+' — pronóstico de un día adelante', xlabel='Fecha objetivo', ylabel='Casos reportados')
    ax.legend(); ax.grid(alpha=.2); fig.autofmt_xdate(); fig.tight_layout()
    fig.savefig(carpeta_graficas/(nombre+'_temporal.png'), dpi=150)
    plt.show(); plt.close(fig)


### Lineal — gráficos de prueba

In [ ]:
# Mismas observaciones de prueba; no se vuelve a entrenar ni se cambia la selección.
graficar_modelo('Lineal', predicciones['Lineal'].to_numpy())


### Ridge — gráficos de prueba

In [ ]:
# Mismas observaciones de prueba; no se vuelve a entrenar ni se cambia la selección.
graficar_modelo('Ridge', predicciones['Ridge'].to_numpy())


### Arbol — gráficos de prueba

In [ ]:
# Mismas observaciones de prueba; no se vuelve a entrenar ni se cambia la selección.
graficar_modelo('Arbol', predicciones['Arbol'].to_numpy())


### RandomForest — gráficos de prueba

In [ ]:
# Mismas observaciones de prueba; no se vuelve a entrenar ni se cambia la selección.
graficar_modelo('RandomForest', predicciones['RandomForest'].to_numpy())


### SVR_lineal — gráficos de prueba

In [ ]:
# Mismas observaciones de prueba; no se vuelve a entrenar ni se cambia la selección.
graficar_modelo('SVR_lineal', predicciones['SVR_lineal'].to_numpy())


### SVR_RBF — gráficos de prueba

In [ ]:
# Mismas observaciones de prueba; no se vuelve a entrenar ni se cambia la selección.
graficar_modelo('SVR_RBF', predicciones['SVR_RBF'].to_numpy())


### RNA_MLP — gráficos de prueba

In [ ]:
# Mismas observaciones de prueba; no se vuelve a entrenar ni se cambia la selección.
graficar_modelo('RNA_MLP', predicciones['RNA_MLP'].to_numpy())


### Persistencia — gráficos de prueba

In [ ]:
# Mismas observaciones de prueba; no se vuelve a entrenar ni se cambia la selección.
graficar_modelo('Persistencia', predicciones['Persistencia'].to_numpy())


### Semanal — gráficos de prueba

In [ ]:
# Mismas observaciones de prueba; no se vuelve a entrenar ni se cambia la selección.
graficar_modelo('Semanal', predicciones['Semanal'].to_numpy())


### Comparación visual y posible sobreajuste
MAE menor es mejor. El panel de entrenamiento/validación muestra medias de folds; las barras de error son desviaciones entre folds, no intervalos de confianza. En COVID, cambios de régimen también pueden ampliar la brecha. La prueba se presenta aparte y no se usa para volver a elegir hiperparámetros.

In [ ]:
# Excluir referencias con métricas train ausentes del panel de brechas.
comp = comparacion_cv.loc[comparacion_cv['area'].ne('Referencia')].copy()
x = np.arange(len(comp))
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].bar(x-.2,comp.MAE_train_CV,.4,label='Entrenamiento CV')
axes[0].bar(x+.2,comp.MAE_val_CV,.4,yerr=comp.MAE_val_std,label='Validación CV',capsize=3)
axes[0].set_xticks(x,comp.modelo,rotation=40,ha='right')
axes[0].set(title='Brecha de generalización',ylabel='MAE'); axes[0].legend()
axes[1].bar(comparacion_test.modelo,comparacion_test.MAE,color='#478b9b')
axes[1].tick_params(axis='x',rotation=60)
axes[1].set(title='MAE de prueba: incluye referencias',ylabel='MAE')
fig.tight_layout(); fig.savefig(carpeta_graficas/'comparacion_MAE.png',dpi=150)
plt.show(); plt.close(fig)


### Fuentes y defensa
Material de clase: *Aprendizaje supervisado — Deep Learning*, Patricia Rodríguez Bilbao, páginas 13–14. Documentación: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html

Explicar: entrada → capas ocultas ReLU → salida continua; Adam y retropropagación; escaladores aprendidos solo en entrenamiento; comparación justa con los otros modelos. MAE y RMSE están en unidades del objetivo, R² puede ser negativo y no es porcentaje de aciertos. No redondeamos predicciones para inventar accuracy. Los valores negativos de COVID se reportan, no se ocultan mediante recorte después de ver resultados.

### Visualizar la recta del modelo lineal múltiple
Mostramos un corte del modelo: variamos una entrada normalizada y mantenemos las demás en sus medianas de entrenamiento. Esta recta sí proviene de los coeficientes aprendidos. No es una nueva regresión simple ni demuestra causalidad. Las combinaciones hipotéticas pueden no representar observaciones reales, especialmente con entradas correlacionadas. Los puntos grises tienen distintas combinaciones de las demás variables, por eso no se espera que coincidan con la recta.


In [ ]:
pre_lineal = modelo_lineal.regressor_[:-1]
est_lineal = modelo_lineal.regressor_.named_steps['modelo']
# transform utiliza parámetros aprendidos en train; no modifica el modelo.
z_train = pre_lineal.transform(X_train)
variable = 'lag_0'
j = X_train.columns.get_loc(variable)
# Variar solo la entrada elegida dentro del intervalo observado en train.
x_recta = np.linspace(z_train[:,j].min(), z_train[:,j].max(), 100)
z_corte = np.tile(np.median(z_train,axis=0), (100,1))
z_corte[:,j] = x_recta
y_recta = est_lineal.predict(z_corte)
# El estimador interno predice y estandarizada; volver a casos reportados.
y_recta = modelo_lineal.transformer_.inverse_transform(y_recta.reshape(-1,1)).ravel()
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(z_train[:,j],y_train,alpha=.2,s=15,label='Datos train (otras entradas varían)')
ax.plot(x_recta,y_recta,color='crimson',linewidth=2.5,label='Corte del modelo: otras entradas en mediana')
ax.set(xlabel=variable+' (entrada normalizada)',ylabel='Objetivo en unidades originales',
       title='Regresión lineal múltiple — corte de la función aprendida')
ax.legend(fontsize=8); ax.grid(alpha=.2); fig.tight_layout()
fig.savefig(carpeta_graficas/'Lineal_recta_condicional.png',dpi=150)
plt.show(); plt.close(fig)


## Lectura de la ejecución verificada de esta entrega
RNA_MLP en prueba: **MAE=610.3064, RMSE=990.9691, R²=0.0920**. Son resultados de esta ejecución con las versiones fijadas; vuelve a ejecutar el notebook para obtener tus tablas. No representan accuracy ni garantizan el mismo rendimiento con datos nuevos.

La red alcanzó 1500 épocas sin satisfacer el criterio de convergencia en el ajuste final y en algunos folds. Es una configuración inicial con limitaciones, no un modelo optimizado. El MAE de train mucho menor que el de validación sugiere problemas de generalización, junto con cambios temporales. Ridge sigue siendo el mejor algoritmo por MAE de validación; Persistencia sigue siendo la mejor solución al incluir referencias. No elegimos un ganador mirando prueba. Una mejora posterior debe usar solo validación temporal; el test ya observado no sirve como nueva confirmación independiente.